## Cell 1: Environment Setup and Imports
#### Subtitle: Initializing the Local Pipeline
#### Description: This cell imports the required libraries for our new dataset approach and verifies that your Lenovo Legion's hardware is correctly utilizing PyTorch.

In [1]:
import os
import torch
import nibabel as nib
import numpy as np
import pandas as pd
from huggingface_hub import login, list_repo_files, hf_hub_download
import matplotlib.pyplot as plt

# Automatically assign the device based on local hardware capabilities
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Pipeline initialized. Utilizing device: {device.upper()}")

Pipeline initialized. Utilizing device: CUDA


## Cell 2: Fetching and Extracting from a Large Archive
#### Subtitle: Downloading a Chunk and Extracting a Single Volume
#### Description: This script targets the exact 12.8 GB archive shown in your second image. It downloads the archive, opens it, extracts only the first 3D NIfTI scan it finds, and then cross-references the extracted filename with the metadata CSV to find the corresponding clinical report.

In [ ]:
import os
import tarfile
import pandas as pd
from huggingface_hub import hf_hub_download

repo_id = "AbdomenAtlas/AbdomenAtlas3.0Mini"

tar_filename = "image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00002553_BDMAP_00002784.tar.gz"

try:
    print(f"1. Downloading {tar_filename}...")
    print("   (WARNING: ~12.8 GB file)")

    # ✅ No token needed if already logged in
    tar_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename=tar_filename
    )

    print("\n2. Extracting a single 3D volume...")

    extracted_dir = "./abdomen_sample"
    os.makedirs(extracted_dir, exist_ok=True)

    target_nifti_path = None

    # ✅ SAFE extraction
    with tarfile.open(tar_path, "r:gz") as tar:
        for member in tar:
            if member.name.endswith(".nii.gz"):
                
                # Get only filename (avoid nested dirs)
                filename = os.path.basename(member.name)
                safe_path = os.path.join(extracted_dir, filename)

                with tar.extractfile(member) as source, open(safe_path, "wb") as dest:
                    dest.write(source.read())

                target_nifti_path = safe_path
                print(f"   Extracted: {filename}")
                break

    if not target_nifti_path:
        raise Exception("No .nii.gz file found in archive.")

    print("\n3. Downloading metadata CSV...")

    csv_path = hf_hub_download(
        repo_id=repo_id,
        repo_type="dataset",
        filename="AbdomenAtlas3.0MiniWithMeta.csv"
    )

    df = pd.read_csv(csv_path)

    base_filename = os.path.basename(target_nifti_path).replace('.nii.gz', '')

    # ✅ FAST vectorized search
    df_str = df.astype(str).apply(lambda row: " ".join(row.values).lower(), axis=1)
    mask = df_str.str.contains(base_filename.lower())

    matched_row = df[mask].iloc[0] if mask.any() else None

    caption = "No matching report found."

    if matched_row is not None:
        for col in df.columns:
            if any(k in col.lower() for k in ['text', 'report', 'finding', 'caption']):
                caption = matched_row[col]
                break

    print("\n--- Success! ---")
    print(f"Local 3D Scan path: {target_nifti_path}")
    print(f"Sample Clinical Caption: {caption}")

    print("\nNote: Archive stored in ~/.cache/huggingface/hub/")

except Exception as e:
    print(f"Error: {e}")

1. Downloading image_only/AbdomenAtlas3_images_BDMAP_BDMAP_00002553_BDMAP_00002784.tar.gz...
   (WARNING: ~12.8 GB file)


image_only/AbdomenAtlas3_images_BDMAP_BD(…):   0%|          | 0.00/12.8G [00:00<?, ?B/s]


2. Extracting a single 3D volume...
   Extracted: ct.nii.gz

3. Downloading metadata CSV...

--- Success! ---
Local 3D Scan path: ./abdomen_sample/ct.nii.gz
Sample Clinical Caption: CT Arterial Phase 

FINDINGS: 
Spleen: 
Normal size (volume: 134.9 cc).
Mean HU value: 124.6 +/- 35.2.

Liver: 
Normal size (volume: 1291.3 cc).
Mean HU value: 111.4 +/- 17.3.

Liver lesions:
Liver lesion 1: 
Location: hepatic segment 2.
Size: 3.1 x 1.6 cm (image 174). Volume: 8.2 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 9.8+/-20.4).

Liver lesion 2: 
Location: hepatic segment 8.
Size: 2.5 x 2.0 cm (image 178). Volume: 4.9 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 36.1+/-31.3).

Liver lesion 3: 
Location: hepatic segment 8.
Size: 1.4 x 0.9 cm (image 170). Volume: 0.7 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 57.8+/-28.2).

Liver lesion 4: 
Location: hepatic segment 1.
Size: 1.1 x 0.7 cm (image 167). Volume: 0.7 cc.
Enhancement relative to l

In [2]:
import os
import pandas as pd
from huggingface_hub import hf_hub_download

# File path
target_nifti_path = "./abdomen_sample/ct.nii.gz"
base_filename = os.path.basename(target_nifti_path).replace('.nii.gz', '')

# Load CSV (cached)
repo_id = "AbdomenAtlas/AbdomenAtlas3.0Mini"
csv_path = hf_hub_download(
    repo_id=repo_id,
    repo_type="dataset",
    filename="AbdomenAtlas3.0MiniWithMeta.csv"
)

df = pd.read_csv(csv_path)

# ⚠️ NOTE: this may not match correctly if filename != BDMAP ID
df_str = df.astype(str).agg(" ".join, axis=1).str.lower()
mask = df_str.str.contains(base_filename.lower())

if mask.any():
    row = df[mask].iloc[0]

    for col in [
        "radiologist note",
        "structured report",
        "narrative report",
        "fusion narrative report"
    ]:
        print(f"\n{'='*60}")
        print(f"{col.upper()}")
        print(f"{'='*60}\n")
        print(row[col])
else:
    print("No matching row found")


RADIOLOGIST NOTE

nan

STRUCTURED REPORT

CT Arterial Phase 

FINDINGS: 
Spleen: 
Normal size (volume: 134.9 cc).
Mean HU value: 124.6 +/- 35.2.

Liver: 
Normal size (volume: 1291.3 cc).
Mean HU value: 111.4 +/- 17.3.

Liver lesions:
Liver lesion 1: 
Location: hepatic segment 2.
Size: 3.1 x 1.6 cm (image 174). Volume: 8.2 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 9.8+/-20.4).

Liver lesion 2: 
Location: hepatic segment 8.
Size: 2.5 x 2.0 cm (image 178). Volume: 4.9 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 36.1+/-31.3).

Liver lesion 3: 
Location: hepatic segment 8.
Size: 1.4 x 0.9 cm (image 170). Volume: 0.7 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 57.8+/-28.2).

Liver lesion 4: 
Location: hepatic segment 1.
Size: 1.1 x 0.7 cm (image 167). Volume: 0.7 cc.
Enhancement relative to liver: Hypoattenuating (HU value is 58.3+/-28.3).

Liver lesion 5: 
Location: hepatic segment 2/3.
Size: 1.1 x 1.1 cm (image 165). Volume: 0.

## Code Cell 3: Volumetric Preprocessing (AbdomenAtlas CT)
#### Subtitle: HU Clipping, 3D Spline Interpolation, and Channel Expansion
#### Description: This cell loads your extracted ct.nii.gz file using nibabel. First, it transposes the array so the depth (Z-axis) is the primary dimension. Then, it applies a targeted abdominal HU clipping window of [-200, 300] to suppress bones and empty air while highlighting the soft tissues (like the liver and spleen) mentioned in your caption. The array is then normalized to a 0.0 to 1.0 floating-point range and downsampled to your local target of $64 \times 128 \times 128$ using third-order spline interpolation. Finally, we duplicate the array across 3 channels to mimic the RGB input required by SigLIP.

In [3]:
import nibabel as nib
import numpy as np
import torch
from scipy.ndimage import zoom

# 1. Load NIfTI file
image_path = "./abdomen_sample/ct.nii.gz"
print(f"Loading 3D NIfTI volume from {image_path}...")

nii_img = nib.load(image_path)
img_data = nii_img.get_fdata().astype(np.float32)

# Convert (X, Y, Z) → (Z, Y, X)
img_data = np.transpose(img_data, (2, 1, 0))
print(f"Raw Volume Shape (Z, Y, X): {img_data.shape}")

# 2. Resize using correct zoom factors on the RAW data first
target_shape = (64, 128, 128)

zoom_factors = (
    target_shape[0] / img_data.shape[0],
    target_shape[1] / img_data.shape[1],
    target_shape[2] / img_data.shape[2],
)

print("Applying 3D interpolation (order=3) to raw HU values...")
# Using order=3 (cubic spline) to preserve soft tissue gradients
resized_data = zoom(img_data, zoom_factors, order=3)  

# 3. HU Clipping & Normalization on the RESIZED data
HU_MIN, HU_MAX = -200, 300
print(f"Clipping HU to [{HU_MIN}, {HU_MAX}] → normalizing to ")

clipped_data = np.clip(resized_data, HU_MIN, HU_MAX)
normalized_data = (clipped_data - HU_MIN) / (HU_MAX - HU_MIN)

# 4. Channel expansion (for SigLIP)
print("Expanding to 3 channels (Z, C, H, W)...")
tensor_3d = np.stack([normalized_data] * 3, axis=1)

# 5. Convert to PyTorch tensor (VERY IMPORTANT)
tensor_3d = torch.from_numpy(tensor_3d).float()

print("\n--- Preprocessing Complete ---")
print(f"Final Tensor Shape: {tensor_3d.shape}")

assert tensor_3d.shape == (64, 3, 128, 128)

Loading 3D NIfTI volume from ./abdomen_sample/ct.nii.gz...
Raw Volume Shape (Z, Y, X): (433, 314, 30)
Applying 3D interpolation (order=3) to raw HU values...
Clipping HU to [-200, 300] → normalizing to 
Expanding to 3 channels (Z, C, H, W)...

--- Preprocessing Complete ---
Final Tensor Shape: torch.Size([64, 3, 128, 128])


## Cell 4: 2D Vision Encoding via SigLIP
#### Subtitle: Slice-by-Slice Feature Extraction
#### Description: This cell loads the pretrained google/siglip-base-patch16-224 vision encoder. To prevent your 8GB laptop GPU from throwing an Out-Of-Memory (OOM) error, we pass the 64 slices through the model in micro-batches of 8.The Hugging Face AutoProcessor will automatically upscale your $128 \times 128$ slices to the $224 \times 224$ resolution expected by the SigLIP base model. Because the patch size is 16, each slice is divided into a $14 \times 14$ grid, yielding 196 discrete patches per slice. We extract the raw last_hidden_state (the un-pooled patch embeddings), resulting in a finalized 3D visual feature matrix of (64, 196, 768).

In [4]:
from transformers import AutoProcessor, SiglipVisionModel
import numpy as np
import torch

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"

model_id = "google/siglip-base-patch16-224"
print(f"Loading {model_id} into VRAM...")

# Load model
processor = AutoProcessor.from_pretrained(model_id)
vision_model = SiglipVisionModel.from_pretrained(model_id).to(device)
vision_model.eval()

batch_size = 8
slice_embeddings = []

print("Initiating slice-by-slice SigLIP feature extraction...")

with torch.no_grad():
    for i in range(0, tensor_3d.shape[0], batch_size):

        # (B, 3, 128, 128)
        batch_slices = tensor_3d[i : i + batch_size]

        # Convert to HWC numpy
        batch_imgs = [
            np.transpose(slice_tensor.cpu().numpy(), (1, 2, 0))
            for slice_tensor in batch_slices
        ]

        # Processor handles resize + normalization
        inputs = processor(images=batch_imgs, return_tensors="pt")
        inputs = inputs.to(device)

        # 🚀 Mixed precision (optional but recommended)
        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            outputs = vision_model(**inputs)

        hidden_states = outputs.last_hidden_state  # (B, 196, 768)

        # Move to CPU to free VRAM
        slice_embeddings.append(hidden_states.cpu())

# Concatenate all slices
E_vision = torch.cat(slice_embeddings, dim=0)

print("\n--- Extraction Complete ---")
print(f"Output Vision Tensor Shape: {E_vision.shape}")

assert E_vision.shape == (64, 196, 768)

Loading google/siglip-base-patch16-224 into VRAM...


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.w

Initiating slice-by-slice SigLIP feature extraction...


/tmp/ipykernel_176229/4027664497.py:38: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device == "cuda")):



--- Extraction Complete ---
Output Vision Tensor Shape: torch.Size([64, 196, 768])


## Cell 5: Slice-Aware Temporal Transformer Adapter (SATTA) 
#### Subtitle: Grouping, Contextualization, and Upscaling
#### Description: This cell defines and executes your novel SATT adapter logic. First, it chunks the 64 slices into groups of 4 and applies mean pooling to compress the depth down to 16 temporal steps. Next, it flattens the temporal and spatial dimensions into a continuous sequence and passes them through a Temporal Transformer, allowing the visual patches to attend to adjacent anatomical slices. Finally, a Multi-Layer Perceptron (MLP) Projector scales the 768-dimensional vision embeddings up to 3072, which is the exact hidden dimension required by the Llama-3.2-3B-Instruct model.

In [5]:
import torch
import torch.nn as nn

# Ensure device is set (fallback to cpu if needed)
device = "cuda" if torch.cuda.is_available() else "cpu"

class SATTAdapter(nn.Module):
    def __init__(self, vision_dim=768, llm_dim=3072, chunk_size=4):
        super().__init__()
        self.chunk_size = chunk_size
        
        # Temporal Transformer to model dependencies across the temporal chunks
        # batch_first=True ensures inputs are (Batch, Sequence, Features)
        encoder_layer = nn.TransformerEncoderLayer(d_model=vision_dim, nhead=8, batch_first=True)
        self.temporal_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # MLP Projector to upscale to Llama-3.2-3B embedding space
        self.projector = nn.Sequential(
            nn.Linear(vision_dim, 2048),
            nn.GELU(),
            nn.Linear(2048, llm_dim)
        )

    def forward(self, x):
        # x shape: (Z_slices, N_patches, D_vision) -> (64, 196, 768)
        Z, N, D = x.shape
        
        # 1. Temporal Grouping / Compression
        # Reshape to (T_chunks, chunk_size, N_patches, D) -> (16, 4, 196, 768)
        T = Z // self.chunk_size
        grouped = x.view(T, self.chunk_size, N, D)
        
        # Apply mean pooling across the chunk dimension
        # Output shape: (16, 196, 768)
        compressed = grouped.mean(dim=1)
        
        # 2. Sequence Flattening
        # Combine temporal and spatial dimensions, add batch dimensiondef forward(self, x):
        # x shape: (Z_slices, N_patches, D_vision) -> (64, 196, 768)
        Z, N, D = x.shape
        
        # 1. Temporal Grouping / Compression
        # Reshape to (T_chunks, chunk_size, N_patches, D) -> (16, 4, 196, 768)
        T = Z // self.chunk_size
        grouped = x.view(T, self.chunk_size, N, D)
        
        # Apply mean pooling across the chunk dimension
        # Output shape: (16, 196, 768)
        compressed = grouped.mean(dim=1)
        
        # 2. Sequence Flattening
        # Combine temporal and spatial dimensions, add batch dimension
        # Output shape: (1, 16 * 196, 768) -> (1, 3136, 768)
        sequence = compressed.view(1, T * N, D)
        
        # 3. Temporal Transformer Contextualization
        contextualized = self.temporal_transformer(sequence)
        
        # 4. Projection to LLM Space
        # Output shape: (1, 3136, 3072)
        llm_tokens = self.projector(contextualized)
        
        return llm_tokens

        # Output shape: (1, 16 * 196, 768) -> (1, 3136, 768)
        sequence = compressed.view(1, T * N, D)
        
        # 3. Temporal Transformer Contextualization
        contextualized = self.temporal_transformer(sequence)
        
        # 4. Projection to LLM Space
        # Output shape: (1, 3136, 3072)
        llm_tokens = self.projector(contextualized)
        
        return llm_tokens

print("Instantiating SATT Adapter...")
satt_adapter = SATTAdapter(vision_dim=768, llm_dim=3072, chunk_size=4).to(device)

# Move the SigLIP vision embeddings to the active device
E_vision_gpu = E_vision.to(device)

print("Passing visual embeddings through the SATT pipeline...")
# Forward pass (simulating the trainable adapter)
visual_tokens = satt_adapter(E_vision_gpu)

print("\n--- SATT Compression Complete ---")
print(f"Final LLM Visual Tokens Shape: {visual_tokens.shape}")
assert visual_tokens.shape == (1, 3136, 3072), "Adapter projection shape mismatch!"

Instantiating SATT Adapter...
Passing visual embeddings through the SATT pipeline...

--- SATT Compression Complete ---
Final LLM Visual Tokens Shape: torch.Size([1, 3136, 3072])


## Cell 6: LLM Integration and "Hello World" Inference
#### Subtitle: 4-bit QLoRA Setup and Multimodal Forward Pass
#### Description: This cell completes the V2L pipeline. It loads the Llama-3.2-3B-Instruct model into your RTX 4060 using 4-bit NormalFloat (NF4) quantization to compress its memory footprint. We then inject Low-Rank Adaptation (LoRA) modules into the attention and MLP layers (q_proj, v_proj, down_proj, etc.) so the model is ready for training. Finally, we concatenate your 3,136 visual tokens with a text prompt and pass them into the LLM to generate a test report.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch

# =========================================================
# DEVICE
# =========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================================
# 4-BIT QUANTIZATION
# =========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# =========================================================
# LOAD LLaMA
# =========================================================

model_id = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

llm = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

device = next(llm.parameters()).device

# =========================================================
# APPLY LoRA
# =========================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj'
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

llm = get_peft_model(llm, lora_config)

llm.print_trainable_parameters()

# =========================================================
# SAFE CAPTION
# =========================================================

caption = globals().get(
    'caption',
    "Multiple hypoattenuating liver masses are present."
)

# =========================================================
# PROMPT
# =========================================================

text_prompt = (
    "<|begin_of_text|>"
    "<|start_header_id|>user<|end_header_id|>\n"
    "Analyze the CT scan and generate a clinical report.\n"
    f"Ground truth hint: {caption}"
    "<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n"
)

inputs = tokenizer(text_prompt, return_tensors="pt").to(device)

# =========================================================
# MULTIMODAL EMBEDDINGS
# =========================================================

print("Preparing multimodal embeddings...")

with torch.no_grad():

    # TEXT EMBEDDINGS
    text_embeds = llm.get_input_embeddings()(inputs.input_ids)

    # VISUAL TOKENS
    # Expected shape: (1, 3136, 3072)

    visual_tokens_gpu = visual_tokens.to(
        dtype=text_embeds.dtype,
        device=device
    )

    # CONCAT VISUAL + TEXT TOKENS

    inputs_embeds = torch.cat(
        [visual_tokens_gpu, text_embeds],
        dim=1
    )

    # ATTENTION MASK

    visual_mask = torch.ones(
        (1, visual_tokens_gpu.shape[1]),
        device=device,
        dtype=inputs.attention_mask.dtype
    )

    attention_mask = torch.cat(
        [visual_mask, inputs.attention_mask],
        dim=1
    )

    # =====================================================
    # GENERATION
    # =====================================================

    print("Running multimodal forward pass...")

    outputs = llm.generate(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        max_new_tokens=75,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

# =========================================================
# DECODE
# =========================================================

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\n--- Pipeline Execution Complete ---")
print(generated_text)

Loading meta-llama/Llama-3.2-3B-Instruct...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511
Preparing multimodal embeddings...
Running multimodal forward pass...

--- Pipeline Execution Complete ---
ingTetSINGI amaltingAldingI haveI amalT
ingTentI AmalTentIveteringIveterIveterIVVII VetIVI.VI VII VII VII VII VII VII VII VII VII VII VII VII VII VII V
